Copyright (C) 2025  Diarmuid

This program is free software: you can redistribute it and/or modify\
it under the terms of the GNU General Public License as published by\
the Free Software Foundation, either version 3 of the License, or\
(at your option) any later version.

This program is distributed in the hope that it will be useful,\
but WITHOUT ANY WARRANTY; without even the implied warranty of\
MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.  See the\
GNU General Public License for more details.

You should have received a copy of the GNU General Public License\
along with this program.  If not, see <https://www.gnu.org/licenses/>.

----------------------------------------------------------------------

Author: Diarmuid O'Neill\
Student Number: C00282898\
Date: 13/01/2026\
Brief Description:
This notebook aims to use KNN Classification to predict if a particular stock in the Dow Jones will be a high mover or a low mover in the next week regardless of direction. Originally an attempt was made to try and predict whether there would be a positive, negative or no gain in a particular stock in the next week. However this was changed to market volatility as it seems to be slightly more predictable using this model.

In [1]:
# Scikit-learn
from sklearn import datasets
from sklearn.metrics import mean_squared_error
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
# other libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from math import sqrt
import seaborn as sns

We start by dropping stock and date columns (non-numerical features), stripping away the '$' character and dropping na values

In [2]:
dataset = pd.read_csv("dow_jones_index.data")
# Pre-processing
dataset = dataset.drop(columns=["stock", "date"])
dataset = dataset.replace('[$]', '', regex=True)
dataset = dataset.dropna()
dataset.head()

,quarter,open,high,low,close,volume,percent_change_price,percent_change_volume_over_last_wk,previous_weeks_volume,next_weeks_open,next_weeks_close,percent_change_next_weeks_price,days_to_next_dividend,percent_return_next_dividend
1,1,16.71,16.71,15.64,15.97,242963398,-4.428490,1.380223,239655616.0,16.19,15.79,-2.470660,19,0.187852
2,1,16.19,16.38,15.60,15.79,138428495,-2.470660,-43.024959,242963398.0,15.87,16.13,1.638310,12,0.189994
3,1,15.87,16.63,15.82,16.13,151379173,1.638310,9.355500,138428495.0,16.18,17.14,5.933250,5,0.185989
4,1,16.18,17.39,16.18,17.14,154387761,5.933250,1.987452,151379173.0,17.33,17.37,0.230814,97,0.175029
5,1,17.33,17.48,16.97,17.37,114691279,0.230814,-25.712195,154387761.0,17.39,17.28,-0.632547,90,0.172712


Below we can see the correlation coefficients for percent_change_next_weeks_price. This gives us an indication as to which features may be good predictors of the target (the closer to 1 the better the predictor). From looking at this, no features are jumping out as great predictors. NOTE: we cannot use next_week_open, next_week_close, percent_change_next_weeks_price, and percent_return_next_dividend as these could not possibly be known beforehand.

In [3]:
# Shows correalation coefficients for predicting percent_change_next_weeks_price
correlation_matrix = dataset.corr()
correlation_matrix["percent_change_next_weeks_price"]

quarter                               0.000098
open                                  0.066695
high                                  0.069589
low                                   0.067255
close                                 0.067789
volume                               -0.091106
percent_change_price                  0.020639
percent_change_volume_over_last_wk    0.013181
previous_weeks_volume                -0.102408
next_weeks_open                       0.067147
next_weeks_close                      0.106578
percent_change_next_weeks_price       1.000000
days_to_next_dividend                -0.028797
percent_return_next_dividend          0.103009
Name: percent_change_next_weeks_price, dtype: float64

Here we will use a selection of features with the highest correlation coefficients. The target is encoded into two distinct classes (high_move and low_move) as knn classification requires discrete classes not continuous variables.

In [39]:
# Small selection of some of the best features
X = dataset[["volume", "previous_weeks_volume", "percent_change_volume_over_last_wk"]]
X = X.values
abs_change = dataset["percent_change_next_weeks_price"].abs() # absolute value (distance from zero regardless of sign)

# Discretization, converting percent_change_next_weeks_price into two distinct classes (high_move, low_move)
threshold = 2.0
y = np.where(
    abs_change >= threshold, "high_move", "low_move"
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=12345
)

In [40]:
# Scaling data so everything is in a comparable range
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [41]:
# Create model
knn_model = KNeighborsClassifier(n_neighbors=3)

In [42]:
# Fit model
knn_model.fit(X_train_scaled, y_train)

,n_neighbors,3
,weights,'uniform'
,algorithm,'auto'
,leaf_size,30
,p,2
,metric,'minkowski'
,metric_params,None
,n_jobs,None


In [43]:
from sklearn.metrics import accuracy_score
# Calculate accuracy scores for KNN classification
train_preds = knn_model.predict(X_train_scaled)
print(f"Training Accuracy (k=3): {accuracy_score(y_train, train_preds):}")
test_preds = knn_model.predict(X_test_scaled)
print(f"Test Accuracy (k=3): {accuracy_score(y_test, test_preds):}")

Training Accuracy (k=3): 0.7430555555555556
Test Accuracy (k=3): 0.5555555555555556


These results imply that the model is not very accurate. It is about 74% accurate for training examples and only 56% accurate for test data. Lets see if we can improve this somewhat through some basic tuning.

In [44]:
from sklearn.model_selection import GridSearchCV
# Tune the model
# Find the optimal number for k (how many neighbours to use for each prediction) (this time for classification)
parameters = {"n_neighbors": range(1, 50)}
gridsearch = GridSearchCV(KNeighborsClassifier(), parameters)
gridsearch.fit(X_train_scaled, y_train)

,estimator,KNeighborsClassifier()
,param_grid,"{'n_neighbors': range(1, 50)}"
,scoring,None
,n_jobs,None
,refit,True
,cv,None
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_neighbors,13


In [45]:
gridsearch.best_params_

{'n_neighbors': 13}

In [46]:
# Add weighted averages based on distance
parameters = {
    "n_neighbors": [13],
    "weights": ["uniform", "distance"],
}
gridsearch = GridSearchCV(KNeighborsClassifier(), parameters)
gridsearch.fit(X_train_scaled, y_train)

,estimator,KNeighborsClassifier()
,param_grid,"{'n_neighbors': [13], 'weights': ['uniform', 'distance']}"
,scoring,None
,n_jobs,None
,refit,True
,cv,None
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_neighbors,13


In [47]:
best_knn = gridsearch.best_params_
# Calculate new accuracy scores for KNN classification after tuning
train_preds_grid = gridsearch.predict(X_train_scaled)
print(f"Training Accuracy (k=13): {accuracy_score(y_train, train_preds_grid):}")
test_preds_grid = gridsearch.predict(X_test_scaled)
print(f"Test Accuracy (k=13): {accuracy_score(y_test, test_preds_grid):}")

Training Accuracy (k=13): 0.6545138888888888
Test Accuracy (k=13): 0.6111111111111112


After finding the optimal value for K (13) there is a little improvement in test accuracy.